In [16]:
from pathlib import Path
from fafbseg import flywire
from meshparty import trimesh_vtk
from multiprocessing import Pool
from tqdm import tqdm

import flybrains
import sys

REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / "code"))

from get_mesh_neuron import *
from cell_groups import *
from color_utils import *
import pickle
import numpy as np 
import pandas as pd 
from make_projection_view_of_3d_rendering import *
import distinctipy

In [33]:
# Load data
Data_ROOT = Path.cwd().resolve().parents[1]

labial_cluster = pd.read_parquet(Data_ROOT/'data/labial_cluster_info_v783.parquet')
g2c = {}
g2c['L1/L2'] = []
g2colors = {}
for t in ['L1','L2','L3','water','high_salt']:#list(dict.fromkeys(labial_cluster.type)):
    # if t not in ['L1','L2','L3']:
    #     g2c[t] = list(labial_cluster.flyid.values[labial_cluster.type==t])
    #     g2colors[t] = list(labial_cluster.color.values[labial_cluster.type==t])[0]
    # else:
    if (t == 'L3')|(t=='high_salt'):
        g2c[t] = list(labial_cluster.flyid.values[labial_cluster.type==t])
        g2colors[t] = '#ff0000'
    elif (t == 'L1')| (t == 'L2'):
        g2c['L1/L2'].extend(list(labial_cluster.flyid.values[labial_cluster.type==t]))
        g2colors['L1/L2'] = '#00ff00'
    else:
        g2c[t] = list(labial_cluster.flyid.values[labial_cluster.type==t])
        g2colors[t] = '#00ff00'







all_cells = list(np.concatenate(list(g2c.values())))

In [34]:
# pool = Pool(4)
# meshes_all = []
# for result in tqdm(pool.imap(load_mesh,all_cells),total=len(all_cells)):
#     meshes_all.append(result)
# pool.close()
# pool.join()

# # Load brain mesh
# available = flywire.get_neuropil_volumes(None)
# neuropil = flybrains.FLYWIRE.mesh
# mesh_ids = [x.id for x in meshes_all]
# meshes_all_navis = navis.NeuronList(meshes_all)
# Save
# with open("neurons.pkl", "wb") as f:
#     pickle.dump(meshes_all_navis, f)

# Load
with open("neurons.pkl", "rb") as f:
    meshes_all_navis = pickle.load(f)
meshes_all = [x for x in meshes_all_navis]

In [35]:
mesh_actors_dict = make_group_mesh_actors(
    meshes=meshes_all,
    g2c=g2c,
    g2color=g2colors,
    neuropil=flybrains.FLYWIRE.mesh,
    cell_opacity=1,
    neuropil_opacity=0.1
)

m = navis.NeuronList(meshes_all)
center = np.array([526232.30530339, 311428.70632942, 128201.97632289])

for g in ['L1/L2','L3','high_salt','water']:
    mesh_actors = [*mesh_actors_dict[g],*mesh_actors_dict['neuropil']]
    
    make_projection_image(
        save_path=f'figures/{g}.png',
        actors_list=mesh_actors,
        center=center,
        projection_view='xy',
        backoff=700,
        parallelsacle=70000,
        do_save=True
    )


mesh_actors = [*mesh_actors_dict['L1/L2'],*mesh_actors_dict['L3'],*mesh_actors_dict['neuropil']]

make_projection_image(
    save_path=f'figures/merge_L1L2_L3.png',
    actors_list=mesh_actors,
    center=center,
    projection_view='xy',
    backoff=700,
    parallelsacle=70000,
    do_save=True
)


mesh_actors = [*mesh_actors_dict['water'],*mesh_actors_dict['high_salt'],*mesh_actors_dict['neuropil']]

make_projection_image(
    save_path=f'figures/merge_water_highsalt.png',
    actors_list=mesh_actors,
    center=center,
    projection_view='xy',
    backoff=700,
    parallelsacle=70000,
    do_save=True
)




2026-08-12 09:25:40.411 (52107.829s) [        74399740]       vtkPNGWriter.cxx:254    ERR| vtkPNGWriter (0x5cb377efaab0): Unable to open file figures/L1/L2.png.png
2026-08-12 09:25:40.411 (52107.829s) [        74399740]     vtkImageWriter.cxx:481    ERR| vtkPNGWriter (0x5cb377efaab0): Ran out of disk space; deleting file(s) already written


In [23]:
mesh_actors_dict = make_group_mesh_actors(
    meshes=meshes_all,
    g2c=g2c,
    g2color=g2colors,
    neuropil=flybrains.FLYWIRE.mesh,
    cell_opacity=1,
    neuropil_opacity=0.1
)

m = navis.NeuronList(meshes_all)
center = np.array([526232.30530339, 311428.70632942, 128201.97632289])

for g in ['L1/L2']:
    mesh_actors = [*mesh_actors_dict[g],*mesh_actors_dict['neuropil']]
    
    make_projection_image(
        save_path=f'figures/L1_l2.png',
        actors_list=mesh_actors,
        center=center,
        projection_view='xy',
        backoff=700,
        parallelsacle=70000,
        do_save=True
    )

In [31]:
mesh_actors_dict.keys()

dict_keys(['L1/L2', 'L3', 'high_salt', 'neuropil'])